# Week 8-B: Generator Draft State and Uncertainty Diagnostics
## Diagnosing Generator Information Needs from Base Draft $y_0$

In DAFiR, routing is **generator-aware**. Before promoting any evidence, we inspect the base draft $y_0$
and extract token-level logits, Shannon entropy, top1-top2 margin, and sequence logprob.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch

REPO_ROOT = Path("/home/gunavenkat/Downloads/CS787-RAG-Project")
SARA_ROOT = REPO_ROOT / "Baselines" / "SARA-main"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import src
if hasattr(src, "__path__") and str(SARA_ROOT / "src") not in src.__path__:
    src.__path__.append(str(SARA_ROOT / "src"))

from src.model.loader import load_sara_for_eval
from src.week8.semantic import SemanticScorer
from src.week8.data import load_config, load_week8_split
from src.week8.draft_generation import execute_resumable_draft_generation
from src.week8.self_consistency import run_self_consistency_sampling
from src.week8.uncertainty import (
    compute_uncertainty_features_for_row,
    evaluate_sufficiency_models,
)

print("Modules imported.")


In [ ]:
cfg = load_config()
split = "internal_val"
records, sfr_dict, _ = load_week8_split(split, cfg)
scorer = SemanticScorer()

semantic_scores_by_qid = {}
for r in records:
    qid = str(r["question_id"])
    sims = scorer.score_query_passages(r["question"], [p["text"] for p in r.get("passages", [])[:10]]).cpu().numpy()
    semantic_scores_by_qid[qid] = sims

device = "cuda" if torch.cuda.is_available() else "cpu"
sara_ckpt_path = REPO_ROOT / cfg["sara"]["checkpoint_path"]
sara_model, sara_tokenizer = load_sara_for_eval(str(sara_ckpt_path), device=device)


In [ ]:
# Generate base draft with diagnostics
draft_cache_path = REPO_ROOT / "results/week8/drafts" / f"draft_diagnostics_{split}.jsonl"
df_drafts = execute_resumable_draft_generation(
    sara_model=sara_model,
    sara_tokenizer=sara_tokenizer,
    records=records,
    semantic_scores_by_qid=semantic_scores_by_qid,
    cached_sfr_embeddings=sfr_dict,
    cache_path=draft_cache_path,
    device=device,
)
print(f"Generated {len(df_drafts)} drafts with token diagnostics.")


In [ ]:
# Extract U1..U12 Uncertainty Features
u_rows = []
for _, row in df_drafts.iterrows():
    u_dict = compute_uncertainty_features_for_row(row.to_dict(), semantic_scorer=scorer)
    u_dict["question_id"] = str(row["question_id"])
    u_rows.append(u_dict)
df_u = pd.DataFrame(u_rows)
print("Uncertainty features computed:")
df_u.describe().T[["mean", "std", "min", "max"]]
